In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression

from sklearn.model_selection import train_test_split, cross_val_score, KFold

from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

## The auto dataset


A dataset with data about 392 different car types and models. The goal is to **predict the car's mpg** based on the other available columns.


In [2]:
path_auto_data = "/content/Auto_ISLR.csv"

In [3]:
auto_df = pd.read_csv(path_auto_data)

In [4]:
auto_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    int64  
 4   weight        392 non-null    int64  
 5   acceleration  392 non-null    float64
 6   year          392 non-null    int64  
 7   origin        392 non-null    int64  
 8   name          392 non-null    object 
dtypes: float64(3), int64(5), object(1)
memory usage: 27.7+ KB


**REPROCESSING**

Remove the 'name' column

In [5]:
auto_df.drop (['name'], axis = 1, inplace = True)

In [6]:
auto_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    int64  
 4   weight        392 non-null    int64  
 5   acceleration  392 non-null    float64
 6   year          392 non-null    int64  
 7   origin        392 non-null    int64  
dtypes: float64(3), int64(5)
memory usage: 24.6 KB


**PREPROCESSING**

Let's fix the 'origin' column

In [7]:
auto_df['origin'].value_counts()

,count
origin,
1,245
3,79
2,68


1 corresponds to 'American', 2 to 'European', and 3 to 'Japanese'

In [8]:
origin_mapper = {1: 'American', 2: 'European', 3: 'Japanese'}

In [9]:
auto_df['origin'] = auto_df['origin'].replace(origin_mapper)

In [10]:
auto_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    int64  
 4   weight        392 non-null    int64  
 5   acceleration  392 non-null    float64
 6   year          392 non-null    int64  
 7   origin        392 non-null    object 
dtypes: float64(3), int64(4), object(1)
memory usage: 24.6+ KB


In [11]:
auto_df['origin'].value_counts()

,count
origin,
American,245
Japanese,79
European,68


**PREPROCESSING**

Now create dummies from 'origin'

In [12]:
auto_df_dummies = pd.get_dummies(auto_df,columns=['origin'], drop_first=True)

In [13]:
auto_df_dummies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   mpg              392 non-null    float64
 1   cylinders        392 non-null    int64  
 2   displacement     392 non-null    float64
 3   horsepower       392 non-null    int64  
 4   weight           392 non-null    int64  
 5   acceleration     392 non-null    float64
 6   year             392 non-null    int64  
 7   origin_European  392 non-null    bool   
 8   origin_Japanese  392 non-null    bool   
dtypes: bool(2), float64(3), int64(4)
memory usage: 22.3 KB


In [14]:
# Variable storing predictors

X_all_auto = auto_df_dummies.drop (['mpg'], axis = 1)

In [15]:
# Variable storing the target

y_auto = auto_df_dummies['mpg']

**1)** Apply the Stats approach using RSE to compare two models:

A linear reqression equation with weight and acceleration as predictors

A linear reqression equation with weight, acceleration, and horsepower as predictors

<br>

Before you start working, you must have the **rse_calculator** function available.

You either import it following the steps I explained in the YouTube video.

Or

You can copy the funtion defition from "Simple_Multiple_LinearReg_Spring26_Part1.ipynb" and paste it here.



**2)** Apply the ML approach to compare the same two models.

Use the chosen model and fit it on the training data.

Then, evaluate its performance on the test data.

**CONTINUE WORKING FROM HERE ON !**

## 1) Statistical Approach: Model Comparison Using RSE

In this section, we compare two linear regression models using the **Residual Standard Error (RSE)**, following the statistical modeling framework discussed in *An Introduction to Statistical Learning*.

The Residual Standard Error measures the typical size of the residuals and is defined as:

\[
\text{RSE} = \sqrt{\frac{\text{RSS}}{n - p - 1}}
\]

where:
- RSS is the residual sum of squares,
- \( n \) is the number of observations,
- \( p \) is the number of predictors.

A **lower RSE** indicates a better-fitting model, after accounting for model complexity.

We compare the following two models:

- **Model 1:** mpg ~ weight + acceleration  
- **Model 2:** mpg ~ weight + acceleration + horsepower  

Both models are fit using the full dataset (no train/test split), which is consistent with the traditional statistical approach to model comparison.


In [16]:
import numpy as np
import statsmodels.api as sm

In [17]:
y = auto_df_dummies['mpg']

### Model 1: mpg ~ weight + acceleration


In [20]:
X1 = auto_df_dummies[['weight', 'acceleration']]
X1 = sm.add_constant(X1)              # adds intercept
results1 = sm.OLS(y, X1).fit()

rse1 = np.sqrt(results1.mse_resid)    # sqrt(RSS/df_resid)
print(f"Model 1 predictors: weight + acceleration -> RSE = {rse1:.4f}")


Model 1 predictors: weight + acceleration -> RSE = 4.2881


## Model 2: mpg ~ weight + acceleration + horsepower


In [21]:
X2 = auto_df_dummies[['weight', 'acceleration', 'horsepower']]
X2 = sm.add_constant(X2)
results2 = sm.OLS(y, X2).fit()

rse2 = np.sqrt(results2.mse_resid)
print(f"Model 2 predictors: weight + acceleration + horsepower -> RSE = {rse2:.4f}")

Model 2 predictors: weight + acceleration + horsepower -> RSE = 4.2456


## Comparison & Interpretation


In [23]:
better = "Model 1" if rse1 < rse2 else "Model 2"
print(f"Lower RSE is better -> {better} wins by RSE.")

Lower RSE is better -> Model 2 wins by RSE.
